
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>

# Demo - Custom Judges with MLflow
## Overview
In this demo, we'll create custom LLM judges using `make_judge()` for evaluating response completeness and tool usage. We'll build both a standard custom judge that evaluates inputs/outputs and a trace-based judge that analyzes the agent's execution flow.

## Learning Objectives
By the end of this demo, you will be able to:
1. Create custom judges with `make_judge()` using natural language instructions and template variables
2. Configure feedback value types for categorical and boolean scoring
3. Build trace-based judges that analyze agent execution workflows
4. Run batch evaluations over traced sessions

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #c62828; font-size: 1.1em;">Prerequisites</strong>
<p style="margin: 8px 0 0 0; color: #333;">This demo uses the agent created in <strong>Demo - Agent Setup</strong>. Please ensure you have completed that notebook before proceeding.</p>
</div>
</div>
</div>

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #c62828; font-size: 1.1em;">Select Compute</strong>
<p style="margin: 8px 0 0 0; color: #333;">Before starting this notebook, select the required compute environment listed below.</p>
<ul style="margin: 12px 0 0 16px; color: #333;">
<li><strong>Serverless Compute, Version 5</strong>: How to select an environment version (<a href="https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">GCP</a>)</li>
</ul>
<p style="margin: 8px 0 0 0; color: #333;"><strong>NOTE:</strong> This notebook was <strong>developed and tested using Serverless V5</strong>. Other compute options may work but are not guaranteed to behave the same or support all features demonstrated.</p>
</div>
</div>
</div>

## A. Classroom Setup
Run the following cell to configure your environment.

In [0]:
%run ../Includes/Classroom-Setup-5

Using Python 3.12.3 environment at: /local_disk0/.ephemeral_nfs/envs/pythonEnv-a5abf189-783b-45ac-9c84-b2be68a05343
Resolved 144 packages in 4.53s
 Downloaded tiktoken
 Downloaded pydantic-core
 Downloaded kiwisolver
 Downloaded zstandard
 Downloaded hf-xet
 Downloaded grpcio
 Downloaded tokenizers
 Downloaded aiohttp
 Downloaded databricks-sdk
 Downloaded cryptography
 Downloaded pillow
 Downloaded mlflow-tracing
 Downloaded fonttools
 Downloaded sqlalchemy
 Downloaded matplotlib
 Downloaded pyarrow
 Downloaded mlflow
 Downloaded scikit-learn
 Downloaded numpy
 Downloaded pandas
 Downloaded scipy
 Downloaded openai
 Downloaded litellm
 Downloaded botocore
Prepared 130 packages in 8.87s
Uninstalled 106 packages in 176ms
Installed 130 packages in 1.85s
 - aiohappyeyeballs==2.4.4
 + aiohappyeyeballs==2.7.1
 - aiohttp==3.11.10
 + aiohttp==3.14.5
 - aiosignal==1.2.0
 + aiosignal==1.4.0
 + alembic==1.20.0
 - annotated-doc==0.0.4
 + annotated-doc==0.0.5
 - annotated-types==0.7.0
 + annotated

Pre-installed Databricks Runtime package(s) replaced: frozenlist: 1.5.0 -> 1.8.0, multidict: 6.1.0 -> 7.1.0, regex: 2024.11.6 -> 2026.9.29, packaging: 24.2 -> 25.0, jsonpointer: 3.0.0 -> 3.3, gitdb: 4.0.11 -> 4.0.12, MarkupSafe: 3.0.2 -> 3.0.4, importlib_metadata: 8.5.0 -> 8.9.0, starlette: 0.50.0 -> 1.9.0, huggingface_hub: 1.2.4 -> 2.2.0, jsonpatch: 1.33 -> 1.35, langsmith: 0.6.1 -> 0.14.11, jmespath: 1.0.1 -> 1.1.0, google-crc32c: 1.8.0 -> 1.9.0, uuid_utils: 0.12.0 -> 0.17.1, google-cloud-core: 2.5.0 -> 2.8.0, orjson: 3.11.5 -> 3.13.0, matplotlib: 3.10.0 -> 3.11.2, tokenizers: 0.22.2 -> 0.23.3, cycler: 0.11.0 -> 0.12.1, idna: 3.7 -> 3.20, cryptography: 44.0.1 -> 46.0.7, grpcio: 1.67.0 -> 1.67.1, scikit-learn: 1.6.1 -> 1.9.1, cachetools: 5.5.1 -> 6.2.6, pytz: 2024.1 -> 2026.5, mypy-extensions: 1.0.0 -> 1.1.0, tzdata: 2024.1 -> 2026.5, langchain-core: 1.2.6 -> 1.6.12, googleapis-common-protos: 1.65.0 -> 1.75.5, annotated-types: 0.7.0 -> 0.8.0, opentelemetry-sdk: 1.39.1 -> 1.45.1, scipy

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Using Python 3.12.3 environment at: /local_disk0/.ephemeral_nfs/envs/pythonEnv-a5abf189-783b-45ac-9c84-b2be68a05343
Resolved 1 package in 204ms
Prepared 1 package in 56ms
Installed 1 package in 1ms
 + backoff==2.2.1
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Using Python 3.12.3 environment at: /local_disk0/.ephemeral_nfs/envs/pythonEnv-a5abf189-783b-45ac-9c84-b2be68a05343
Resolved 129 packages in 2.06s
 Downloaded openai-agents
 Downloaded databricks-connect
 Downloaded mlflow-tracing
 Downloaded mlflow-skinny
 Downloaded mlflow
Prepared 28 packages in 1.33s
Uninstalled 11 packages in 43ms
Installed 29 packages in 254ms
 + aiohttp-retry==2.9.1
 - blinker==1.7.0
 ~ blinker==1.9.0
 + databricks-ai-bridge==0.22.0
 + databricks-ai-search==0.79
 - databricks-connect==18.1.9
 + databricks-con

Pre-installed Databricks Runtime package(s) replaced: googleapis-common-protos: 1.65.0 -> 1.75.0, protobuf: 5.29.4 -> 5.29.6, databricks-connect: 18.1.9 -> 17.0.10, mlflow-skinny: 3.8.1 -> 3.17.1, PyJWT: 2.10.1 -> 2.15.1, wcwidth: 0.2.5 -> 0.9.2. uv pip install may report the previous version as uninstalled, but the runtime's preinstalled packages remain on disk.


Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Using Python 3.12.3 environment at: /local_disk0/.ephemeral_nfs/envs/pythonEnv-a5abf189-783b-45ac-9c84-b2be68a05343
Resolved 43 packages in 1.17s
Prepared 1 package in 79ms
Uninstalled 1 package in 0.57ms
Installed 1 package in 1ms
 - databricks-vectorsearch==0.75
 + databricks-vectorsearch==0.68
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


Starting Demo Environment Setup
Using config: /Workspace/Users/labuser17704131_1792356850@vocareum.com/genai-agent-evaluation-on-databricks-en_us-1.1.1/Agent Evaluation on Databricks/Includes/config/config_5.yaml

[1] Setting up catalog...
Vocareum workspace detected.
  Catalog 'labuser17704131_1792356850' found. Using it.

[2] Setting up schema...
  Catalog : labuser17704131_1792356850
  Schema  : demo_agent_eval
Refreshing environment

Demo Environment Setup Complete


If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc


<Experiment: artifact_location='dbfs:/databricks/mlflow-tracking/4107611117782955', creation_time=1792358017483, effective_trace_archival_retention=None, experiment_id='4107611117782955', last_update_time=1792365553922, lifecycle_stage='active', name='/Users/labuser17704131_1792356850@vocareum.com/airbnb_experiment', tags={'mlflow.experiment.sourceName': '/Users/labuser17704131_1792356850@vocareum.com/airbnb_experiment',
 'mlflow.experimentKind': 'genai_development',
 'mlflow.experimentType': 'MLFLOW_EXPERIMENT',
 'mlflow.ownerEmail': 'labuser17704131_1792356850@vocareum.com',
 'mlflow.ownerId': '71082494983411'}, trace_location=None, workspace='default'>

2026/10/19 00:00:39 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environment:
 - mlflow (current: 3.17.1, required: mlflow==3.13.0)
To fix the mismatches, call `mlflow.pyfunc.get_model_dependencies(model_uri)` to fetch the model's environment and install dependencies using the resulting environment file.
/local_disk0/.ephemeral_nfs/envs/pythonEnv-a5abf189-783b-45ac-9c84-b2be68a05343/lib/python3.12/site-packages/databricks/connect/session.py:476: UserWarning: Ignoring the default notebook Spark session and creating a new Spark Connect session. To use the default notebook Spark session, use DatabricksSession.builder.getOrCreate() with no additional parameters.
  warnings.warn(new_notebook_session_msg)


Custom Judge Endpoint: databricks:/databricks-gpt-5-mini
Coherent Instructions: Evaluate if the response is coherent, maintaining a constant tone
and following a clear flow of thoughts/concepts.
Question: {{ inputs }}
Response: {{ outputs }}

Tool Usage Instructions: Analyze the {{ trace }} to verify correct tool usage.

Check that the agent:
- Called tools with correct and reasonable parameters
- Used tool outputs effectively to generate the final response
- Did not make unnecessary or redundant tool calls

Return "correct" if tool usage was appropriate and effective,
"incorrect" if tools were misused or inappropriate tools were selected.



In [0]:
print(f"Username:          {username}")
print(f"Catalog Name:      {catalog_name}")
print(f"Schema Name:       {schema_name}")

Username:          labuser17704131_1792356850@vocareum.com
Catalog Name:      labuser17704131_1792356850
Schema Name:       demo_agent_eval


## B. Load Evaluation Dataset

The `custom_eval.json` dataset contains queries without expectations. We'll use it to evaluate the agent's response completeness.

In [0]:
import json
from pathlib import Path
from pprint import pprint

path = Path(f"/Volumes/{catalog_name}/{schema_name}/agent_vol/custom_eval.json")

with path.open("r", encoding="utf-8") as f:
    custom_eval = json.load(f)

pprint(custom_eval)

[{'inputs': {'input': [{'content': 'How many Entire home/apt listings are in '
                                   'the Mission neighborhood?',
                        'role': 'user'}]}},
 {'inputs': {'input': [{'content': 'Count the number of Private room listings '
                                   'in Nob Hill.',
                        'role': 'user'}]}},
 {'inputs': {'input': [{'content': 'What is the average listing price in '
                                   'Haight Ashbury?',
                        'role': 'user'}]}}]


## C. Standard Custom Judge - Response Completeness

We'll use `make_judge()` to create a judge that evaluates whether responses are complete. The `instructions` parameter uses the `coherent_instructions` loaded from the agent eval config, which references `{{inputs}}` and `{{outputs}}` template variables.

The `feedback_value_type` is set to a `Literal` type so the judge returns one of `"complete"`, `"partial"`, or `"incomplete"`.

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 14px 18px; border-radius: 4px; margin: 16px 0;">
  <strong style="display:block; color:#0d47a1; margin-bottom:6px; font-size: 1.1em;">Why metrics will be empty</strong>
  <div style="color:#333;">Because the feedback type is a <code>Literal</code> (categorical), MLflow cannot compute a numeric mean — so <code>metrics</code> will be <code>{}</code>. Boolean feedback types (like in section D) <em>do</em> produce a mean metric.</div>
</div>

In [0]:
from typing import Literal
from mlflow.genai import make_judge

response_completeness_feedback_value_type = Literal["complete", "partial", "incomplete"]

completeness_judge = make_judge(
    name="response_completeness",
    instructions=coherent_instructions,
    feedback_value_type=response_completeness_feedback_value_type
)

### C1. Run Completeness Evaluation

In [0]:
completeness_results = mlflow.genai.evaluate(
    data=custom_eval,
    predict_fn=lambda input: agent.predict({"input": input}),
    scorers=[completeness_judge]
)

2026/10/19 00:02:59 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.
2026/10/19 00:02:59 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


Evaluating:   0%|          | 0/3 [Elapsed: 00:00, Remaining: ?]

### C2. Inspect Results

In [0]:
print(f"The run ID is: {completeness_results.run_id}")
print(f"The aggregated metrics are: {completeness_results.metrics}")
print("\nThe results from the completeness evaluation:")
display(completeness_results.result_df)

The run ID is: c0dba860c89f45eaaa2cd13904c95a20
The aggregated metrics are: {}

The results from the completeness evaluation:


trace_id response_completeness/value response_completeness/rationale trace client_request_id state request_time execution_duration request response trace_metadata tags spans assessments tr-0b9ea4e856babe10a570f9a1be9f36fd complete The response is coherent and maintains a consistent tone throughout. It clearly answers the user's query about the number of 'Entire home/apt' listings in the Mission neighborhood by providing the exact number (6) and reiterating it in a clear statement. The flow of information is logical, starting from the function call to retrieve the data, followed by the output of that function, and concluding with a direct response to the user. There are no abrupt changes in tone or structure, making the response easy to follow. {"info": {"trace_id": "tr-0b9ea4e856babe10a570f9a1be9f36fd", "client_request_id": "tr-0b9ea4e856babe10a570f9a1be9f36fd", "trace_location": {"type": "MLFLOW_EXPERIMENT", "mlflow_experiment": {"experiment_id": "4107611117782955"}}, "request_time": "2026-10-19T00:03:08.755Z", "state": "OK", "trace_metadata": {"mlflow.sourceRun": "c0dba860c89f45eaaa2cd13904c95a20", "mlflow.trace_schema.version": "3", "mlflow.databricks.workspaceID": "7474659189051797", "mlflow.trace.tokenUsage": "{\"input_tokens\": 710, \"output_tokens\": 164, \"total_tokens\": 874, \"cache_read_input_tokens\": 672}", "mlflow.modelId": "m-76cbb6e9978c4d179ff19d82411101ef", "mlflow.databricks.notebookID": "4107611117782941", "mlflow.databricks.notebookPath": "/Users/labuser17704131_1792356850@vocareum.com/genai-agent-evaluation-on-databricks-en_us-1.1.1/Agent Evaluation on Databricks/Module 2 - Built-In and Guideline Judges/2.6 Demo - Custom Judges with MLflow", "mlflow.databricks.workspaceURL": "https://dbc-4d4ce156-516a.cloud.databricks.com", "mlflow.source.name": "/Users/labuser17704131_1792356850@vocareum.com/genai-agent-evaluation-on-databricks-en_us-1.1.1/Agent Evaluation on Databricks/Module 2 - Built-In and Guideline Judges/2.6 Demo - Custom Judges with MLflow", "mlflow.traceOutputs": "{\"tool_choice\": null, \"truncation\": null, \"id\": null, \"created_at\": null, \"error\": null, \"incomplete_details\": null, \"instructions\": null, \"metadata\": null, \"model\": null, \"object\": \"response\", \"output\": [{\"type\": \"function_call\", \"id\": \"chatcmpl_...", "mlflow.traceInputs": "{\"request\": {\"tool_choice\": null, \"truncation\": null, \"max_output_tokens\": null, \"metadata\": null, \"parallel_tool_calls\": null, \"tools\": null, \"reasoning\": null, \"store\": null, \"stream\": null, \"temperature\": null, \"text\": null, \"top_p\": null, \"use...", "mlflow.trace.sizeStats": "{\"total_size_bytes\": 29918, \"num_spans\": 5, \"max\": 10238, \"p25\": 2322, \"p50\": 4203, \"p75\": 10089}", "mlflow.source.type": "NOTEBOOK", "mlflow.databricks.notebook.commandID": "1001792367512955_7429208434228412973_1402b085a2a349f298c8d376f21040a4", "mlflow.user": "spark-a5abf189-783b-45ac-9c84-b2", "mlflow.trace.sizeBytes": "29918", "mlflow.databricks.webappURL": "https://dbc-4d4ce156-516a.cloud.databricks.com"}, "assessments": [{"assessment_id": "a-01704bccc7674f6facf03fe9121d7fd9", "assessment_name": "response_completeness", "trace_id": "tr-0b9ea4e856babe10a570f9a1be9f36fd", "span_id": "c13beb7a6210e55d", "source": {"source_type": "LLM_JUDGE", "source_id": "databricks"}, "create_time": "2026-10-19T00:03:18.490Z", "last_update_time": "2026-10-19T00:03:18.490Z", "feedback": {"value": "complete"}, "rationale": "The response is coherent and maintains a consistent tone throughout. It clearly answers the user's query about the number of 'Entire home/apt' listings in the Mission neighborhood by providing the exact number (6) and reiterating it in a clear statement. The flow of information is logical, starting from the function call to retrieve the data, followed by the output of that function, and concluding with a direct response to the user. There are no abrupt changes in tone or structure, making the response easy to follow.",

## D. Trace-Based Custom Judge - Tool Usage

Trace-based judges analyze the full execution trace using `{{trace}}` in the instructions. This judge validates whether the agent used tools correctly.

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 14px 18px; border-radius: 4px; margin: 16px 0;">
  <strong style="display:block; color:#0d47a1; margin-bottom:6px; font-size: 1.1em;">How trace-based judges work</strong>
  <div style="color:#333;">When instructions contain <code>{{trace}}</code>, the judge receives the full trace object and uses tool calling to inspect span data (inputs, outputs, tool calls, timing). The <code>model</code> parameter is optional.</div>
</div>

In [0]:
tool_usage_judge = make_judge(
    name="tool_usage_validator",
    instructions=tool_usage_instructions,
    feedback_value_type=bool,
    model=custom_eval_endpoint
)

### D1. Generate Traces

To evaluate tool usage, we first need traces. The helper function below queries the agent and wraps each call in an MLflow span with session metadata, then we search for those traces.

In [0]:
from typing import Tuple, Dict, Any
from mlflow.entities import SpanType

def gen_trace_data(query: str, model: str = custom_eval_endpoint, user_id: str = username, session_id: str = session_id) -> Tuple[Dict[str, Any]]:
    with mlflow.start_span(name="populate_agent_trace", span_type=SpanType.AGENT) as span:
        mlflow.update_current_trace(
            metadata={
                "mlflow.trace.session": session_id,
                "mlflow.trace.user": user_id
            },
            tags={
                "training_type": "agent_eval_training",
                "model": model,
                "agent_type": "TOOL-CALLING"
            }
        )

        query_payload = [
            {"input": [{"role": "user", "content": query}]}]

        response = agent.predict(query_payload)

        span.set_inputs({"query": query})
        span.set_outputs({"response": response})
        trace_id = span.trace_id

    return query, trace_id

In [0]:
queries = [
    "How many Entire home/apt listings are in the Mission neighborhood?",
    "Count the number of Private room listings in Nob Hill.",
    "What is the average listing price in Haight Ashbury?"
]

for query in queries:
    gen_trace_data(query=query)

[Trace(trace_id='tr-9ab615632fa4d1f251d3b762a7c81c71'), Trace(trace_id='tr-5644a3dde1266272c8b1618b0b74aa87'), Trace(trace_id='tr-476ebdb376938a60c7ca2aca315a872d')]

### D2. Evaluate a Single Trace

Search for the traces we just generated using the session ID, then evaluate the first one individually.

In [0]:
session_traces = mlflow.search_traces(
    locations=[experiment_id],
    filter_string=f"metadata.`mlflow.trace.session` = '{session_id}'",
    return_type="list")
session_traces[0]

Trace(trace_id='tr-476ebdb376938a60c7ca2aca315a872d')

In [0]:
trace = session_traces[0]
feedback = tool_usage_judge(trace=trace)

The judge returns a `Feedback` object with a boolean `value` (pass/fail) and a `rationale` explaining its reasoning.

In [0]:
print(f"Assessment: {feedback.value}")
print(f"Rationale: {feedback.rationale}")

Assessment: True
Rationale: The agent used tools appropriately: it called the tool that lists spans, retrieved span metadata, then fetched the root span to inspect inputs and outputs, and also retrieved the specific tool span. Calls used correct parameters (max_results for list_spans, span_id and attributes_to_fetch for get_span). The tool outputs were used to confirm the function call, its arguments (neighborhood_name: Haight Ashbury), the function output (88.71428571428571), and the assistant’s final message. There were no unnecessary or redundant calls—only the relevant span-listing and span-fetching operations were performed—so tool usage is correct and effective.


### D3. Batch Trace Evaluation

Evaluate all session traces at once using `mlflow.genai.evaluate()`.

In [0]:
trace_judge_results = mlflow.genai.evaluate(
    data=session_traces,
    scorers=[tool_usage_judge]
)

Evaluating:   0%|          | 0/3 [Elapsed: 00:00, Remaining: ?]


### D4. Inspect Batch Results

Review the aggregated metrics and per-trace results from the batch evaluation. Because `feedback_value_type=bool`, MLflow can compute a mean — `1.0` means all traces passed.

In [0]:
print(f"The run ID is: {trace_judge_results.run_id}")
print(f"The aggregated metrics are: {trace_judge_results.metrics}")
print("\nThe results from trace-based evaluation:")
display(trace_judge_results.result_df)

The run ID is: ea5575e4b4ba4c648eab40de55407bac
The aggregated metrics are: {'tool_usage_validator/mean': np.float64(1.0)}

The results from trace-based evaluation:


trace_id tool_usage_validator/value tool_usage_validator/rationale trace client_request_id state request_time execution_duration request response trace_metadata tags spans assessments tr-476ebdb376938a60c7ca2aca315a872d true The agent used tools appropriately: it called the trace-info tool to get high-level metadata, listed spans to see available spans, and fetched the root and the specific tool span to inspect inputs/outputs. Parameters passed to each tool were correct and minimal (list_spans with max_results, get_span with span_ids). The tool outputs were used to confirm the function call (avg_neigh_price) with argument "Haight Ashbury" and its numeric output (88.71428571428571), which matches the assistant message. There were no unnecessary or redundant calls—the sequence (get_trace_info -> list_spans -> get_span root -> get_span tool) is logical and sufficient to verify correctness. {"info": {"trace_id": "tr-476ebdb376938a60c7ca2aca315a872d", "client_request_id": "tr-476ebdb376938a60c7ca2aca315a872d", "trace_location": {"type": "MLFLOW_EXPERIMENT", "mlflow_experiment": {"experiment_id": "4107611117782955"}}, "request_time": "2026-10-19T00:11:23.096Z", "state": "OK", "trace_metadata": {"mlflow.databricks.workspaceID": "7474659189051797", "mlflow.trace_schema.version": "3", "mlflow.databricks.notebookPath": "/Users/labuser17704131_1792356850@vocareum.com/genai-agent-evaluation-on-databricks-en_us-1.1.1/Agent Evaluation on Databricks/Module 2 - Built-In and Guideline Judges/2.6 Demo - Custom Judges with MLflow", "mlflow.databricks.notebookID": "4107611117782941", "mlflow.databricks.workspaceURL": "https://dbc-4d4ce156-516a.cloud.databricks.com", "mlflow.source.name": "/Users/labuser17704131_1792356850@vocareum.com/genai-agent-evaluation-on-databricks-en_us-1.1.1/Agent Evaluation on Databricks/Module 2 - Built-In and Guideline Judges/2.6 Demo - Custom Judges with MLflow", "mlflow.traceOutputs": "{\"response\": {\"object\": \"response\", \"output\": [{\"type\": \"function_call\", \"id\": \"chatcmpl_e04d8149-e328-4826-b714-1030df44e213_8b37a6dd-85dd-4c74-91c0-d16a5b48f157\", \"call_id\": \"call_a1475a7e-43d0-4689-b68a-92700e8dae2c\", \"name\": \"labuser17704131_1...", "mlflow.traceInputs": "{\"query\": \"What is the average listing price in Haight Ashbury?\"}", "mlflow.trace.sizeStats": "{\"total_size_bytes\": 11315, \"num_spans\": 4, \"max\": 4133, \"p25\": 1190, \"p50\": 1839, \"p75\": 2751}", "mlflow.source.type": "NOTEBOOK", "mlflow.databricks.notebook.commandID": "1001792367512955_7429208434228412973_1402b085a2a349f298c8d376f21040a4", "mlflow.trace.user": "labuser17704131_1792356850@vocareum.com", "mlflow.user": "spark-a5abf189-783b-45ac-9c84-b2", "mlflow.trace.session": "custom-eval-001", "mlflow.trace.sizeBytes": "11315", "mlflow.databricks.webappURL": "https://dbc-4d4ce156-516a.cloud.databricks.com"}, "assessments": [{"assessment_id": "a-648b75c1de2549928886193b64a90630", "assessment_name": "tool_usage_validator", "trace_id": "tr-476ebdb376938a60c7ca2aca315a872d", "span_id": "a6a7a4b438a695ff", "source": {"source_type": "LLM_JUDGE", "source_id": "databricks:/databricks-gpt-5-mini"}, "create_time": "2026-10-19T00:48:22.683Z", "last_update_time": "2026-10-19T00:48:22.683Z", "feedback": {"value": true}, "rationale": "The agent used tools appropriately: it called the trace-info tool to get high-level metadata, listed spans to see available spans, and fetched the root and the specific tool span to inspect inputs/outputs. Parameters passed to each tool were correct and minimal (list_spans with max_results, get_span with span_ids). The tool outputs were used to confirm the function call (avg_neigh_price) with argument \"Haight Ashbury\" and its numeric output (88.71428571428571), which matches the assistant message. There were no unnecessary or redundant calls\u2014the sequence (get_trace_info -> list_spans -> get_span root -> get_span tool) is logical and sufficient to verify correctness.", "metadata": {"mlflow.assessment.judg

## E. Conclusion
In this demo, you:

1. Created a standard custom judge with `make_judge()` using `Literal` feedback types
2. Built a trace-based judge that analyzes agent tool usage via `{{trace}}`
3. Generated traces with session metadata and evaluated them individually and in batch

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>

